# Handling Missing Values

Real-world datasets often include missing values (empty fields, `NaN`, `None`). How you handle them depends on *why* they’re missing and how much information is affected.

This notebook uses `../Datasets/missing_data.csv`, where some fields are blank (e.g., Category/Sub-Category/Profit/Quantity). When loaded with `pandas.read_csv`, those blank entries are treated as missing (`NaN`).

## Goals
- Detect missing values and quantify them per column/row
- Decide whether to drop, fill, or interpolate based on context
- Apply a consistent handling strategy
- Validate that the result still makes sense for analysis/modeling

## Quick detection toolkit (pandas)
```python
df.isna().sum().sort_values(ascending=False)     # missing count per column
df.isna().mean().mul(100).round(2)               # missing % per column
df[df.isna().any(axis=1)].head()                 # rows with at least one missing value
```

## Common strategies
**1) Drop missing**
```python
df_drop_rows = df.dropna()                             # drop rows with any missing values
df_drop_cols = df.dropna(axis=1, thresh=len(df)*0.8)   # keep cols with >=80% non-missing
```

**2) Simple imputation (`fillna`)**
```python
df2 = df.copy()
df2["Profit"] = df2["Profit"].fillna(df2["Profit"].median())
df2["Category"] = df2["Category"].fillna(df2["Category"].mode().iat[0])
```

**3) Forward/Backward fill (often for ordered/time-like data)**
```python
df2 = df.sort_values("Customer ID").copy()
df2["Sub-Category"] = df2["Sub-Category"].ffill().bfill()
```

**4) Group-wise imputation (keeps structure)**
```python
df2 = df.copy()
df2["Profit"] = df2["Profit"].fillna(df2.groupby("Category")["Profit"].transform("median"))
```

## Import Packages

In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

## Read The Dataset

In [ ]:
df = pd.read_csv('../Datasets/missing_data.csv')

## Initial Data Exploration

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dtypes

## Identify Missing Values

In [ ]:
df.isna().head()

In [ ]:
# Check entire dataframe
df.isna().any().any()

In [ ]:
# Check each Column
df.isna().any()

In [ ]:
# Find rows with Missing Values
df.isna().any(axis=1)

## Calculate Missing Values & Missing Percentages

In [ ]:
# Count Missing Values by Column
df.isna().sum()

In [ ]:
# Count Rows Containing Missing Values
df.isna().any(axis=1).sum()

In [ ]:
# Calculate Missing Percentages by Column
df.isna().mean().mul(100).round(2)

In [ ]:
# Create Missing Values by Summary
missing_summary = pd.DataFrame({'Missing Values': df.isna().sum(), 'Missing Percentage': df.isna().mean().mul(100).round(2)})

In [ ]:
missing_summary

In [ ]:
## Find Complete Rows

In [ ]:
df_non_missing = df[df.notna().all(axis=1)]

In [ ]:
df_non_missing.shape

## Drop Rows with Missing Values

In [ ]:
df_drop = df.dropna()

In [ ]:
print(f"Original Dataset {df.shape} No Null Dataset has {df_drop.shape}")

In [ ]:
print(df.shape[0] - df_drop.shape[0])

## Drop Missing Values from Specific Columns

In [ ]:
df_sales = df.dropna(subset=['Sales'])

In [ ]:
df_sales.isna().mean() * 100

In [ ]:
df_sales.shape

In [ ]:
df_multi = df.dropna(subset = ['Sales', 'Profit'])

In [ ]:
df_multi.isna().sum()

In [ ]:
df_multi.shape

## how = 'any' and how = 'all'

In [ ]:
df.dropna(how = 'all', subset = ['Sales', 'Profit'])

## Imputing Numerical Variables with Mean, Median, or a Constant

In [ ]:
sales_mean = df['Sales'].mean()

In [ ]:
sales_mean

In [ ]:
df_numerical = df.copy()

In [ ]:
df_numerical['Sales'] = df_numerical['Sales'].fillna(sales_mean)

In [ ]:
df_numerical.isna().sum()

In [ ]:
profit_meadian = df_numerical['Profit'].median()

In [ ]:
profit_meadian

In [ ]:
df_numerical['Profit'] = df_numerical['Profit'].fillna(profit_meadian)

In [ ]:
df_numerical.isna().sum()

In [ ]:
df_numerical["Avg Discount"] = df_numerical["Avg Discount"].fillna(0)

In [ ]:
df_numerical.isna().sum()

## Imputing Categorical Variables with Mode and Constant

In [ ]:
df.isna().sum()

In [ ]:
df_string = df.copy()

In [ ]:
category_mode = df_string['Category'].mode()[0]

In [ ]:
category_mode

In [ ]:
df_string['Category'] = df_string['Category'].fillna(category_mode)

In [ ]:
df_string.isna().sum()

In [ ]:
df_string["Sub-Category"] = df_string['Category'].fillna('Not Available')

In [ ]:
df_string["Sub-Category"].value_counts()

## Forward Fill and Backward Fill

In [ ]:
df.tail(100)

In [ ]:
df_fill = df.copy()

In [ ]:
df_fill['Sales'] = df_fill['Sales'].ffill()

In [ ]:
df_fill.tail(100)

In [ ]:
df_fill['Profit'] = df_fill['Profit'].bfill()

In [ ]:
df_fill.tail(100)